In [ ]:
# Simple, clean approach with the unified interface
from soros_system.core.portfolio_analyzer import PortfolioAnalyzer
from scripts.assetsRoster import carteira_AC, carteira_HB, others, carteira_EXC, carteira_LC
import warnings

warnings.filterwarnings('ignore')

#assets = carteira_AC
assets = set(carteira_AC+carteira_HB+carteira_EXC+carteira_LC+others)
#assets=['bitcoin', 'ethereum', 'solana', 'dogecoin', 'chainlink', 'pendle']

analyzer = PortfolioAnalyzer( 
    data_dir='/Users/valter.rebelo/MissionControl/data',
    data_path='/Users/valter.rebelo/MissionControl/data/micro/candleData/',
    btc_data_path='/Users/valter.rebelo/MissionControl/data/micro/candleData/bitcoin_candles.csv',
    ssr_data_path='/Users/valter.rebelo/MissionControl/data/onchainData/BTC_SSR.csv',
    market_data_path='/Users/valter.rebelo/MissionControl/data/micro/assetData/',
    asset_ids=assets

)

# Load assets - the analyzer handles paths internally

analyzer.load_data(assets=assets, start_date='2014-01-01', end_date='2025-12-31')





In [ ]:
# Register specific signals by their exact names
signals = analyzer.register_signals_parallel(
    asset_ids=assets, 
    calculate_values=True,
    specific_signals=[
                      "DonchianEnsembleUSD", "DonchianEnsembleBTC",
                      "RSI_Bullish_USD", "RSI_Bullish_BTC"]
)
signals


In [ ]:
# Select exactly which signals you want to use
usd_signals = [
   
    'DonchianEnsembleUSD',
    'RSI_Bullish_USD',
    #'RSI_Ensemble_Signal_USD'
]
 
btc_signals = [
    #'RSI_Oversold_BTC',
    #'RSI_Bullish_BTC'
    #'BullRegimeSignalBTC'
    #'RSI_Ensemble_Signal_BTC'
]



# Run the backtest for all assets at once
results = analyzer.backtest_assets(
    asset_ids=assets,
    start_date="2023-1-1",
    end_date="2025-12-31",
    initial_capital=10000.0,
    btc_cost=0.001,  # Lower cost for Bitcoin
    alt_cost=0.005,  # Higher cost for altcoins
    usd_signals=usd_signals,
    btc_signals=btc_signals,
    use_btc_filter=True
)


In [ ]:
# Filter for results where strategy outperforms buy & hold, sorted by outperformance
outperformance_df = results['summary'].copy()
outperformance_df['Outperformance'] = outperformance_df['Total Return (%)'] - outperformance_df['Buy & Hold Return (%)']
# Filter for positive outperformance and positive total return
# Also exclude stablecoins like USDT, USDC, TUSD
filtered_df = outperformance_df[
    ((outperformance_df['Outperformance'] > 0) | 
    (outperformance_df['Total Return (%)'] > 0)) &
    (~outperformance_df.index.isin(['USDT', 'USDC', 'TUSD']))
]
filtered_df.sort_values('Outperformance', ascending=False)[:60]


In [ ]:
# Filter out assets where buy & hold Sharpe ratio is larger than the strategy's Sharpe ratio
results['asset_results'].get('fartcoin')['trades']

In [ ]:
# Get the results dataframe for Bitcoin
bitcoin_results = results['asset_results'].get('nunet')['results_df']

# Import plotly
import plotly.graph_objects as go

# Create a plotly figure
fig = go.Figure()

# Add traces for portfolio value and buy & hold value
fig.add_trace(
    go.Scatter(
        x=bitcoin_results.index,
        y=bitcoin_results['portfolio_value'],
        mode='lines',
        name='SOROS V1',
        line=dict(width=2, color='blue')
    )
)

fig.add_trace(
    go.Scatter(
        x=bitcoin_results.index,
        y=bitcoin_results['buy_hold_value'],
        mode='lines',
        name='Buy & Hold',
        line=dict(width=2, color='red')
    )
)

# Update layout
fig.update_layout(
    title='SOROS vs. Buy n Hold',
    xaxis_title='',
    yaxis_title='Valor da Carteira (USD)',
    legend=dict(x=0.01, y=0.99),
    hovermode='x unified',
    template='plotly_white'
)

# Show the figure
fig.show()

In [ ]:
results['asset_results'].get('nunet')['metrics_comparison']

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Read the CSV file
etf = pd.read_csv('/Users/valter.rebelo/MissionControl/ETF flows - Sheet1.csv')

# Replace all '-' values with 0
etf = etf.replace('-', 0)

# Convert columns to numeric where possible
for col in etf.columns:
    if col != 'Date':  # Skip the Date column
        etf[col] = pd.to_numeric(etf[col], errors='coerce')

# Convert Date to datetime
etf['Date'] = pd.to_datetime(etf['Date'], format='%d %b %Y')
etf.set_index('Date', inplace=True)

# Get BTC price data
btc_data = analyzer.get_asset_data('bitcoin').price_data
btc_close = btc_data['close'].to_frame('BTC_Price')

# Merge ETF flows with BTC price data
merged_data = etf.join(btc_close, how='left')

# Resample to weekly data
# For ETF flows, sum the daily flows
# For BTC price, take the last price of the week
weekly_etf = etf.resample('W').sum()
weekly_btc = btc_close.resample('W').last()

# Calculate weekly BTC returns
weekly_btc['BTC_Return'] = weekly_btc['BTC_Price'].pct_change() * 100

# Merge weekly ETF flows with weekly BTC returns
weekly_data = weekly_etf.join(weekly_btc, how='left')

# Create a scatter plot with regression line
plt.figure(figsize=(10, 6))

# Get all data points except the two most recent ones
main_data = weekly_data.iloc[:-2]
# Get the most recent data point
most_recent = weekly_data.iloc[-1:]
# Get the second most recent data point
second_most_recent = weekly_data.iloc[-2:-1]

# Plot main data points
sns.regplot(x='Total', y='BTC_Return', data=main_data, 
            scatter_kws={'alpha':0.6, 'color':'blue'}, 
            line_kws={'color':'red'})

# Plot the most recent data point with a different color
plt.scatter(most_recent['Total'], most_recent['BTC_Return'], 
            color='green', s=100, alpha=0.8, label='Dado Mais Recente')

# Plot the second most recent data point with a different color
plt.scatter(second_most_recent['Total'], second_most_recent['BTC_Return'], 
            color='purple', s=100, alpha=0.8, label='Segundo Dado Mais Recente')

plt.title('Retornos do BTC e Fluxos de ETFs Semanais')
plt.xlabel('Fluxos de ETFs (Total)')
plt.ylabel('Retorno do BTC (%)')
plt.grid(True, alpha=0.3)
plt.legend()

# Calculate correlation and regression
correlation = weekly_data['Total'].corr(weekly_data['BTC_Return'])
slope, intercept, r_value, p_value, std_err = stats.linregress(
    weekly_data['Total'].dropna(), 
    weekly_data['BTC_Return'].dropna()
)

# Add correlation and regression stats to the plot
plt.annotate(f'Correlation: {correlation:.3f}\nR-squared: {r_value**2:.3f}\np-value: {p_value:.4f}', 
             xy=(0.05, 0.85), xycoords='axes fraction', 
             bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))

plt.tight_layout()
plt.show()

# Display the data
weekly_data

In [43]:
asset_id = 'myria'

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# Get price data
asset_data = analyzer.get_asset_data(asset_id).price_data
close_prices = asset_data['close']

# Determine if we should show BTC charts based on asset_id
show_btc_charts = asset_id != 'bitcoin'

# Create subplots with appropriate number of rows
if show_btc_charts:
    # Show both USD and BTC charts (4 rows)
    fig = make_subplots(rows=4, cols=1, 
                        subplot_titles=("Donchian Breakout Model BTC", 
                                       "Donchian Breakout Model USD", 
                                       "RSI Bullish BTC",
                                       "RSI Bullish USD"),
                        vertical_spacing=0.1,
                        shared_xaxes=True)
    num_rows = 4
else:
    # Only show USD charts (2 rows)
    fig = make_subplots(rows=2, cols=1, 
                        subplot_titles=("Breakout Model", 
                                        "Momentum Model"),
                        vertical_spacing=0.1,
                        shared_xaxes=True)
    num_rows = 2

# Add background price line with low alpha to all charts
for i in range(1, num_rows + 1):
    fig.add_trace(go.Scatter(
        x=close_prices.index, 
        y=close_prices, 
        mode='lines', 
        name='Price',
        line=dict(color='gray', width=1),
        opacity=0.2,
        showlegend=False if i > 1 else True
    ), row=i, col=1)

# Current row tracker for USD-only mode
current_row = 1

# 1. Donchian Ensemble BTC (only if not bitcoin)
if show_btc_charts:
    values_btc = analyzer.get_asset_data(asset_id).get_signal('DonchianEnsembleBTC').values
    bull_btc = values_btc == 1
    bear_btc = values_btc == 0

    green_column_btc = pd.Series(np.nan, index=close_prices.index)
    red_column_btc = pd.Series(np.nan, index=close_prices.index)

    green_column_btc[bull_btc] = close_prices[bull_btc]
    red_column_btc[bear_btc] = close_prices[bear_btc]

    fig.add_trace(go.Scatter(
        x=green_column_btc.index, 
        y=green_column_btc, 
        mode='lines', 
        name='Bullish BTC (1)',
        line=dict(color='green')
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=red_column_btc.index, 
        y=red_column_btc, 
        mode='lines', 
        name='Bearish/Neutral BTC (0)',
        line=dict(color='red')
    ), row=1, col=1)
    
    current_row += 1

# 2. Donchian Ensemble USD
values_usd = analyzer.get_asset_data(asset_id).get_signal('DonchianEnsembleUSD').values
bull_usd = values_usd == 1
bear_usd = values_usd == 0

green_column_usd = pd.Series(np.nan, index=close_prices.index)
red_column_usd = pd.Series(np.nan, index=close_prices.index)

green_column_usd[bull_usd] = close_prices[bull_usd]
red_column_usd[bear_usd] = close_prices[bear_usd]

fig.add_trace(go.Scatter(
    x=green_column_usd.index, 
    y=green_column_usd, 
    mode='lines', 
    name='Apetite por Risco',
    line=dict(color='green'),
    showlegend=current_row == 1  # Only show in legend if it's the first row
), row=current_row, col=1)

fig.add_trace(go.Scatter(
    x=red_column_usd.index, 
    y=red_column_usd, 
    mode='lines', 
    name='Aversão por Risco',
    line=dict(color='red'),
    showlegend=current_row == 1  # Only show in legend if it's the first row
), row=current_row, col=1)

current_row += 1

# 3. RSI Bullish BTC (only if not bitcoin)
if show_btc_charts:
    rsi_bullish_btc = analyzer.get_asset_data(asset_id).get_signal('RSI_Bullish_BTC').values
    
    bull_rsi_btc = rsi_bullish_btc == 1
    not_bull_rsi_btc = ~bull_rsi_btc

    green_column_rsi_btc = pd.Series(np.nan, index=close_prices.index)
    red_column_rsi_btc = pd.Series(np.nan, index=close_prices.index)

    green_column_rsi_btc[bull_rsi_btc] = close_prices[bull_rsi_btc]
    red_column_rsi_btc[not_bull_rsi_btc] = close_prices[not_bull_rsi_btc]

    fig.add_trace(go.Scatter(
        x=green_column_rsi_btc.index, 
        y=green_column_rsi_btc, 
        mode='lines', 
        name='RSI Bullish BTC (1)',
        line=dict(color='green'),
        showlegend=True
    ), row=current_row, col=1)

    fig.add_trace(go.Scatter(
        x=red_column_rsi_btc.index, 
        y=red_column_rsi_btc, 
        mode='lines', 
        name='Not RSI Bullish BTC (0)',
        line=dict(color='red'),
        showlegend=True
    ), row=current_row, col=1)
    
    current_row += 1

# 4. RSI Bullish USD
rsi_bullish_usd = analyzer.get_asset_data(asset_id).get_signal('RSI_Bullish_USD').values

bull_rsi_usd = rsi_bullish_usd == 1
not_bull_rsi_usd = ~bull_rsi_usd

green_column_rsi_usd = pd.Series(np.nan, index=close_prices.index)
red_column_rsi_usd = pd.Series(np.nan, index=close_prices.index)

green_column_rsi_usd[bull_rsi_usd] = close_prices[bull_rsi_usd]
red_column_rsi_usd[not_bull_rsi_usd] = close_prices[not_bull_rsi_usd]

fig.add_trace(go.Scatter(
    x=green_column_rsi_usd.index, 
    y=green_column_rsi_usd, 
    mode='lines', 
    name='RSI Bullish USD (1)',
    line=dict(color='green'),
    showlegend=current_row == 3  # Only show in legend if it's the first row in USD-only mode
), row=current_row, col=1)

fig.add_trace(go.Scatter(
    x=red_column_rsi_usd.index, 
    y=red_column_rsi_usd, 
    mode='lines', 
    name='Not RSI Bullish USD (0)',
    line=dict(color='red'),
    showlegend=current_row == 3  # Only show in legend if it's the first row in USD-only mode
), row=current_row, col=1)

# Update layout
fig.update_layout(
    title=f'SOROS - {asset_id.capitalize()}',
    xaxis_title='',
    legend_title='Diagnóstico',
    height=800 if asset_id == 'bitcoin' else 1600,  # Adjust height based on number of charts
    width=1000,
    template='plotly_white'
)

# Update y-axis titles for all rows
for i in range(1, num_rows + 1):
    fig.update_yaxes(title_text="Preço (USD)", row=i, col=1)

# Show the plot
fig.show()


In [ ]:
results['asset_results'].get(asset_id)['metrics_comparison']

In [ ]:

import pandas as pd
def display_all_assets_signals_for_date(analyzer, date_str):
    """
    Display Donchian Ensemble, RSI, and Variance signals for all assets on a specific date
    
    Args:
        analyzer: PortfolioAnalyzer instance
        date_str: Date string in 'YYYY-MM-DD' format
    
    Returns:
        DataFrame with signal values for all assets on the specified date
    """
    # Convert date string to datetime
    date = pd.to_datetime(date_str)
    
    # List of signals to check
    signals_to_check = [
        #'DonchianEnsembleBTC',
        #'DonchianEnsembleUSD',
        'RSI_Bullish_BTC',
        'RSI_Bullish_USD',
        #'BullHighVarianceSignalBTC',
        #'BullHighVarianceSignalUSD',
        #'BullLowVarianceSignalBTC',
       # 'BullLowVarianceSignalUSD'
    ]
    
    # Get all asset IDs
    asset_ids = sorted(list(analyzer.assets.keys()))
    
    # Create empty DataFrame with assets as rows and signals as columns
    result_df = pd.DataFrame(index=asset_ids, columns=signals_to_check)
    
    # Fill the DataFrame with signal values
    for asset_id in asset_ids:
        asset_data = analyzer.get_asset_data(asset_id)
        if asset_data is None:
            continue
            
        for signal_name in signals_to_check:
            try:
                signal_data = asset_data.get_signal(signal_name)
                if signal_data is not None:
                    # Access the signal values directly
                    # SignalData objects typically have a 'values' attribute that is a pandas Series
                    if hasattr(signal_data, 'values') and isinstance(signal_data.values, pd.Series):
                        signal_series = signal_data.values
                        if date in signal_series.index:
                            result_df.loc[asset_id, signal_name] = signal_series.loc[date]
                        else:
                            result_df.loc[asset_id, signal_name] = None
                    else:
                        result_df.loc[asset_id, signal_name] = None
                else:
                    result_df.loc[asset_id, signal_name] = None
            except Exception as e:
                print(f"Error getting {signal_name} for {asset_id} on {date_str}: {e}")
                result_df.loc[asset_id, signal_name] = None
    
    # Replace NaN with None for cleaner display
    result_df = result_df.fillna("N/A")
    
    # Add a sum column that counts the number of active signals (value of 1)
    result_df['Signal_Sum'] = result_df.apply(
        lambda row: sum(1 for val in row if val == 1), 
        axis=1
    )

    result_df = result_df.sort_index()
    
    return result_df

# Example usage - replace with your desired date
date_to_check = "2025-5-11"   # Change to your desired date

signals_df = display_all_assets_signals_for_date(analyzer, date_to_check)
print(f"Signals for all assets on {date_to_check}:")
display(signals_df.tail(60))


In [ ]:
def plot_signal_sums_through_time(analyzer, start_date=None, end_date=None, min_market_cap=1000000000, max_market_cap=None):
    """
    Create an interactive plotly lollipop chart showing signal sums for all assets through time.
    Assets are on the Y-axis and dates on the X-axis.
    
    Args:
        analyzer: The PortfolioAnalyzer instance
        start_date: Optional start date string in format 'YYYY-MM-DD'. If None, uses earliest available date.
        end_date: Optional end date string in format 'YYYY-MM-DD'. If None, uses latest available date.
        min_market_cap: Minimum market capitalization in USD to include an asset
        max_market_cap: Maximum market capitalization in USD to include an asset
    
    Returns:
        DataFrame with signal sums over time
    """
    import plotly.graph_objects as go
    import pandas as pd
    from soros_system.data.data_loader import DataLoader
    
    # Get all asset IDs
    asset_ids = list(analyzer.assets.keys())
    
    # Exclude stablecoins
    excluded_assets = ['tether', 'usd-coin']
    asset_ids = [asset_id for asset_id in asset_ids if asset_id not in excluded_assets]
    
    # Create a data loader to get tickers
    data_loader = DataLoader(
        data_path=analyzer.data_path,
        btc_data_path=analyzer.btc_data_path,
        ssr_data_path=analyzer.ssr_data_path,
        market_data_path=analyzer.market_data_path,
        asset_ids=analyzer.asset_ids
    )
    
    # Create a mapping from asset_id to ticker
    ticker_mapping = {}
    for asset_id in asset_ids:
        ticker = data_loader.get_ticker_from_id(asset_id)
        ticker_mapping[asset_id] = ticker.upper() if ticker else asset_id.upper()
    
    # Create a dictionary to store market caps
    market_caps = {}
    
    # Filter assets by market cap if specified
    if min_market_cap > 0 or max_market_cap is not None:
        filtered_asset_ids = []
        for asset_id in asset_ids:
            asset_data = analyzer.get_asset_data(asset_id)
            if asset_data is not None:
                # Try different ways to access market cap
                market_cap = None
                
                # Try to get from metadata if available
                if hasattr(asset_data, 'metadata') and asset_data.metadata:
                    market_cap = asset_data.metadata.get('market_cap')
                
                # Try to get from asset_data directly
                if market_cap is None and hasattr(asset_data, 'market_cap'):
                    market_cap = asset_data.market_cap
                
                # Try to get from price_data if available
                if market_cap is None and hasattr(asset_data, 'price_data'):
                    if 'market_cap' in asset_data.price_data.columns:
                        # Use the most recent market cap value
                        market_cap = asset_data.price_data['market_cap'].iloc[-1]
                
                # Store market cap for later sorting
                market_caps[asset_id] = market_cap if market_cap is not None else 0
                
                # Debug print to see what we're working with
                ticker = ticker_mapping.get(asset_id, asset_id)
                print(f"Asset: {ticker}, Market Cap: {market_cap}")
                
                # Check if market cap meets criteria
                meets_min_criteria = market_cap is None or market_cap >= min_market_cap
                meets_max_criteria = max_market_cap is None or (market_cap is not None and market_cap <= max_market_cap)
                
                # Include asset if market cap meets criteria
                if meets_min_criteria and meets_max_criteria:
                    filtered_asset_ids.append(asset_id)
                    if market_cap is None:
                        print(f"Warning: No market cap data for {ticker}, including it anyway")
        
        asset_ids = filtered_asset_ids
        
        if not asset_ids:
            market_cap_msg = f"with market cap >= {min_market_cap}"
            if max_market_cap is not None:
                market_cap_msg += f" and <= {max_market_cap}"
            print(f"No assets found {market_cap_msg}")
            return pd.DataFrame()
    else:
        # If no market cap filtering, still collect market caps for sorting
        for asset_id in asset_ids:
            asset_data = analyzer.get_asset_data(asset_id)
            if asset_data is not None:
                market_cap = None
                
                if hasattr(asset_data, 'metadata') and asset_data.metadata:
                    market_cap = asset_data.metadata.get('market_cap')
                
                if market_cap is None and hasattr(asset_data, 'market_cap'):
                    market_cap = asset_data.market_cap
                
                if market_cap is None and hasattr(asset_data, 'price_data'):
                    if 'market_cap' in asset_data.price_data.columns:
                        market_cap = asset_data.price_data['market_cap'].iloc[-1]
                
                market_caps[asset_id] = market_cap if market_cap is not None else 0
    
    # Determine date range
    if start_date is None or end_date is None:
        # Find the common date range across assets
        min_dates = []
        max_dates = []
        for asset_id in asset_ids:
            asset_data = analyzer.get_asset_data(asset_id)
            if asset_data is not None and hasattr(asset_data, 'price_data') and len(asset_data.price_data) > 0:
                min_dates.append(asset_data.price_data.index[0])
                max_dates.append(asset_data.price_data.index[-1])
        
        if not min_dates or not max_dates:
            print("No data available for any assets")
            return
        
        if start_date is None:
            start_date = max(min_dates).strftime('%Y-%m-%d')
        if end_date is None:
            end_date = min(max_dates).strftime('%Y-%m-%d')
    
    # Convert to datetime objects
    start_dt = pd.to_datetime(start_date)
    end_dt = pd.to_datetime(end_date)
    
    # Generate a list of dates - daily intervals as requested
    date_range = pd.date_range(start=start_dt, end=end_dt, freq='D')
    date_strings = [date.strftime('%Y-%m-%d') for date in date_range]
    
    # Create a DataFrame to store signal sums for each asset and date
    signal_sums_over_time = pd.DataFrame(index=asset_ids)
    
    # Calculate signal sums for each date
    for date_str in date_strings:
        signals_df = display_all_assets_signals_for_date(analyzer, date_str)
        signal_sums_over_time[date_str] = signals_df['Signal_Sum']
    
    # Add market cap column for sorting
    signal_sums_over_time['market_cap'] = signal_sums_over_time.index.map(lambda x: market_caps.get(x, 0))
    
    # Sort by market cap in descending order
    signal_sums_over_time = signal_sums_over_time.sort_values(by='market_cap', ascending=True)
    
    # Calculate total signal sum (but don't include market_cap column)
    signal_sums_over_time['total_sum'] = signal_sums_over_time.drop(columns=['market_cap']).sum(axis=1)
    
    # Remove the sorting columns after sorting
    signal_sums_over_time = signal_sums_over_time.drop(columns=['market_cap', 'total_sum'])
    
    # Create a new DataFrame with tickers as index
    ticker_signal_sums = pd.DataFrame(index=[ticker_mapping[asset_id] for asset_id in signal_sums_over_time.index])
    for col in signal_sums_over_time.columns:
        ticker_signal_sums[col] = signal_sums_over_time[col].values
    
    # Create color mapping function based on signal sum (scaled from 0 to 2)
    def get_color(signal_sum):
        if signal_sum == 0:
            return 'rgb(180, 50, 50)'  # darker red
        elif signal_sum == 1:
            return 'rgb(220, 180, 50)'  # darker yellow
        else:  # 2 signals
            return 'rgb(50, 150, 50)'  # darker green
    
    # Create the figure
    fig = go.Figure()
    
    # Get all dates for x-axis
    all_dates = pd.to_datetime(ticker_signal_sums.columns)
    
    # Plot all points including zero values
    for ticker in ticker_signal_sums.index:
        ticker_data = ticker_signal_sums.loc[ticker]
        
        # Create lists to store data for this ticker
        x_dates = []
        y_tickers = []
        sizes = []
        colors = []
        hover_texts = []
        
        # Process each date point
        for date, sum_val in zip(all_dates, ticker_data):
            x_dates.append(date)
            y_tickers.append(ticker)
            sizes.append(10 + sum_val * 5)  # Size based on signal sum
            colors.append(get_color(sum_val))
            hover_texts.append(f"Date: {date.strftime('%Y-%m-%d')}<br>Ticker: {ticker}<br>Signal Sum: {sum_val}")
        
        # Add scatter plot (dots)
        fig.add_trace(go.Scatter(
            x=x_dates,
            y=y_tickers,
            mode='markers',
            marker=dict(
                size=sizes,
                color=colors,
                line=dict(width=1, color='rgba(0,0,0,0.5)')
            ),
            name=ticker,
            showlegend=False,
            hoverinfo='text',
            hovertext=hover_texts
        ))
    
    # Add a color legend
    for i, (sum_val, color_desc) in enumerate([
        (0, "0 signals (red)"),
        (1, "1 signal (yellow)"),
        (2, "2 signals (green)")
    ]):
        fig.add_trace(go.Scatter(
            x=[None],
            y=[None],
            mode='markers',
            marker=dict(size=10 + sum_val * 5, color=get_color(sum_val)),
            name=color_desc,
            showlegend=True
        ))
    
    # Update layout
    title_text = "Signal Sums for Assets Through Time (Ordered by Market Cap)"
    if min_market_cap > 0 or max_market_cap is not None:
        title_text += f" (Market Cap"
        if min_market_cap > 0:
            title_text += f" >= {min_market_cap:,} USD"
        if max_market_cap is not None:
            if min_market_cap > 0:
                title_text += " and"
            title_text += f" <= {max_market_cap:,} USD"
        title_text += ")"
        
    fig.update_layout(
        title=title_text,
        xaxis_title="Date",
        yaxis_title="Tickers",
        height=max(800, len(ticker_signal_sums) * 20),  # Dynamic height based on number of assets
        width=1200,  # Fixed width for better display
        yaxis=dict(
            categoryorder='array',
            categoryarray=ticker_signal_sums.index,  # Order by market cap (already sorted)
        ),
        xaxis=dict(
            type='date',
            tickformat='%Y-%m-%d',
            tickangle=45,
        ),
        legend_title="Signal Sum Legend",
        hovermode="closest",
        showlegend=True,
        margin=dict(l=150, r=50, t=50, b=100)  # Add margin for ticker names
    )
    
    fig.show()
    
    return ticker_signal_sums

# Example usage
signal_data_over_time = plot_signal_sums_through_time(
    analyzer, 
    start_date="2025-05-1",  # Change to your desired start date
    end_date="2025-06-3",    # Change to your desired end date
    min_market_cap=00000000,
    max_market_cap= 100000000

)

In [ ]:

btc = results['asset_results'].get('bitcoin')['results_df']
btc

In [ ]:
import pandas as pd
import numpy as np

def apply_triple_barrier_labeling(df, profit_factor=2, stop_factor=2, time_window=7, stdev_window=20, debug=False):
    """
    Apply triple barrier method labeling to a trading dataframe with the following rules:
    - Positive Label (+1): Upper barrier (profit target) is breached first
    - Negative Label (-1): Lower barrier (stop loss) is breached first
    - Neutral Label (0): Time barrier expires without hitting either barrier
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The dataframe containing trading data
    profit_factor : float, default=2
        Multiplier for standard deviation to determine profit target
    stop_factor : float, default=2
        Multiplier for standard deviation to determine stop loss
    time_window : int, default=7
        Number of days to look forward for label determination
    stdev_window : int, default=20
        Window size for standard deviation calculation
    debug : bool, default=False
        Whether to print debugging information
        
    Returns:
    --------
    pandas.DataFrame
        The original dataframe with additional 'triple_barrier_label' column
    """
    # Make a copy of the dataframe to avoid modifying the original
    df_copy = df.copy()
    
    # Calculate rolling standard deviation
    df_copy['rolling_std'] = df_copy['close'].rolling(window=stdev_window).std()
    
    # Initialize the label column
    df_copy['triple_barrier_label'] = np.nan
    
    # Debugging counters
    if debug:
        skipped_beginning = 0
        skipped_end = 0
        processed = 0
        labels_positive = 0
        labels_negative = 0
        labels_neutral = 0
    
    # Find rows where position changes (actual trade entries)
    position_entries = []
    
    # Look for either:
    # 1. Where position changes from 0 to non-zero
    if 'position' in df_copy.columns:
        for i in range(1, len(df_copy)):
            if df_copy['position'].iloc[i-1] == 0 and df_copy['position'].iloc[i] != 0:
                position_entries.append(i)
    
    # 2. Or where trade_executed is True
    if 'trade_executed' in df_copy.columns:
        for i in range(len(df_copy)):
            if df_copy['trade_executed'].iloc[i] == True:
                if i not in position_entries:
                    position_entries.append(i)
    
    # 3. Or where final_decision_shifted is 1 (assuming this indicates a trade signal)
    for i in range(len(df_copy)):
        if not pd.isna(df_copy['final_decision_shifted'].iloc[i]) and df_copy['final_decision_shifted'].iloc[i] == 1:
            if i not in position_entries:
                position_entries.append(i)
    
    if debug:
        print(f"Found {len(position_entries)} potential trade entry points")
    
    # Loop through each identified trade entry
    for i in position_entries:
        # Skip if we're too close to the beginning or end of the dataframe
        if i < stdev_window:
            if debug: skipped_beginning += 1
            continue
            
        if i >= len(df_copy) - 1:
            if debug: skipped_end += 1
            continue
        
        # Calculate profit target and stop loss
        entry_price = df_copy['open'].iloc[i]
        std_dev = df_copy['rolling_std'].iloc[i-1]  # Using previous day's std for decision
        
        # Skip if std_dev is NaN or 0
        if pd.isna(std_dev) or std_dev == 0:
            if debug: print(f"Warning: Standard deviation is {std_dev} at index {i}")
            continue
        
        profit_target = entry_price + profit_factor * std_dev
        stop_loss = entry_price - stop_factor * std_dev
        
        # Look forward up to time_window days
        label = None
        exit_day = min(i+time_window, len(df_copy)-1)
        
        for forward_idx in range(i+1, exit_day+1):
            current_price = df_copy['close'].iloc[forward_idx]
            
            # Check if profit target is hit first
            if current_price >= profit_target:
                label = 1  # Positive label
                if debug: labels_positive += 1
                break
                
            # Check if stop loss is hit first
            if current_price <= stop_loss:
                label = -1  # Negative label
                if debug: labels_negative += 1
                break
        
        # If neither barrier is hit within the time window, assign neutral label
        if label is None:
            label = 0  # Neutral label - time barrier hit first
            if debug: labels_neutral += 1
        
        # Assign the label
        df_copy.loc[df_copy.index[i], 'triple_barrier_label'] = label
        
        if debug: processed += 1
    
    if debug:
        print(f"Skipped (beginning): {skipped_beginning}")
        print(f"Skipped (end): {skipped_end}")
        print(f"Processed entries: {processed}")
        print(f"Positive labels (+1): {labels_positive}")
        print(f"Negative labels (-1): {labels_negative}")
        print(f"Neutral labels (0): {labels_neutral}")
        print(f"Non-NaN labels: {df_copy['triple_barrier_label'].notna().sum()}")
    
    return df_copy

# Example usage:
labeled_df = apply_triple_barrier_labeling(btc)
labeled_df

In [ ]:
# Import plotly if not already imported
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create a figure with two subplots stacked vertically
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1, 
                    subplot_titles=('BTC Price', 'Trading Signals'))

# Plot BTC price in the top subplot
fig.add_trace(
    go.Scatter(x=labeled_df.index, y=labeled_df['close'], name='BTC Price', line=dict(color='blue')),
    row=1, col=1
)

# Plot position and triple barrier label in the bottom subplot
fig.add_trace(
    go.Scatter(x=labeled_df.index, y=labeled_df['position'], name='Position', line=dict(color='green')),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=labeled_df.index, y=labeled_df['triple_barrier_label'], name='Triple Barrier Label', 
               line=dict(color='red')),
    row=2, col=1
)

# Update layout
fig.update_layout(
    height=800, 
    width=1200,
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Update y-axis labels
fig.update_yaxes(title_text="Price (USD)", row=1, col=1)
fig.update_yaxes(title_text="Signal Value", row=2, col=1)
fig.update_xaxes(title_text="Date", row=2, col=1)

# Add grid lines
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGrey')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGrey')

# Show the figure
fig.show()

In [ ]:
def create_returns_df_from_signals(analyzer, signal_df, min_days=90, min_market_cap=1000000000, max_market_cap=None):
    """
    Create a DataFrame of returns for assets that have a signal sum >= 3
    and have at least the minimum number of days of price data.
    
    Args:
        analyzer: PortfolioAnalyzer instance
        signal_df: DataFrame with signal values and Signal_Sum column
        min_days: Minimum number of days of price data required (default: 365)
        min_market_cap: Minimum market capitalization in USD to include an asset (default: 0)
        max_market_cap: Maximum market capitalization in USD to include an asset (default: None)
    
    Returns:
        DataFrame with daily returns for qualifying assets
    """
    # Filter assets with signal sum >= 3
    # For bitcoin, we only require a signal sum of 2, for all other assets we require 3
    bitcoin_assets = signal_df[
        (signal_df.index == 'bitcoin') & (signal_df['Signal_Sum'] >= 2)
    ].index.tolist()
    other_assets = signal_df[
        (signal_df.index != 'bitcoin') & (signal_df['Signal_Sum'] >= 4)
    ].index.tolist()
    qualified_assets = bitcoin_assets + other_assets
    
    # Dictionary to store returns series
    returns_dict = {}
    
    # Process each qualified asset
    for asset_id in qualified_assets:
        asset_data = analyzer.get_asset_data(asset_id)
        
        # Skip if asset data is None
        if asset_data is None:
            continue
            
        # Check market cap if filtering is enabled
        if min_market_cap > 0 or max_market_cap is not None:
            market_cap = None
            
            # Try to get from metadata if available
            if hasattr(asset_data, 'metadata') and asset_data.metadata:
                market_cap = asset_data.metadata.get('market_cap')
            
            # Try to get from asset_data directly
            if market_cap is None and hasattr(asset_data, 'market_cap'):
                market_cap = asset_data.market_cap
            
            # Try to get from price_data if available
            if market_cap is None and hasattr(asset_data, 'price_data'):
                if 'market_cap' in asset_data.price_data.columns:
                    # Use the most recent market cap value
                    market_cap = asset_data.price_data['market_cap'].iloc[-1]
            
            # Skip if market cap doesn't meet criteria
            if market_cap is not None:
                if (min_market_cap > 0 and market_cap < min_market_cap) or \
                   (max_market_cap is not None and market_cap > max_market_cap):
                    print(f"Skipping {asset_id}: Market cap {market_cap} outside range [{min_market_cap}, {max_market_cap or 'inf'}]")
                    continue
            elif min_market_cap > 0:
                print(f"Skipping {asset_id}: No market cap data available")
                continue
            
        # Get price data
        try:
            price_data = asset_data.get_price_data()
            
            if price_data is None or len(price_data) < min_days:
                print(f"Skipping {asset_id}: Only has {len(price_data) if price_data is not None else 0} days of data (minimum required: {min_days})")
                continue
                
            if 'close' in price_data.columns:
                # Calculate daily returns
                returns = price_data['close'].pct_change().dropna()
                
                # Make sure the index is unique before adding to the dictionary
                if returns.index.is_unique:
                    returns_dict[asset_id] = returns
                else:
                    # Handle duplicate indices by keeping the first occurrence
                    print(f"Warning: {asset_id} has duplicate dates. Keeping first occurrence.")
                    returns = returns[~returns.index.duplicated(keep='first')]
                    returns_dict[asset_id] = returns
            else:
                print(f"Skipping {asset_id}: Price data missing 'close' column")
        except Exception as e:
            print(f"Error processing {asset_id}: {e}")
    
    # Create DataFrame from the dictionary
    if returns_dict:
        # Create an empty DataFrame with a unique index that spans all series
        all_dates = sorted(set().union(*[returns.index for returns in returns_dict.values()]))
        returns_df = pd.DataFrame(index=all_dates)
        
        # Add each series to the DataFrame
        for asset_id, returns in returns_dict.items():
            returns_df[asset_id] = returns
            
        print(f"Created returns DataFrame with {len(returns_df.columns)} assets")
        return returns_df
    else:
        print("No qualifying assets found")
        return pd.DataFrame()

# Example usage
# Get signals for a specific date
date_to_check = "2024-11-25"  # Change to your desired date
signals_df = display_all_assets_signals_for_date(analyzer, date_to_check)

# Create returns DataFrame for assets with signal sum >= 3
returns_df = create_returns_df_from_signals(analyzer, signals_df)

# Display the first few rows of the returns DataFrame
if not returns_df.empty:
    display(returns_df.tail())
    
    # Display basic statistics
    print(f"\nReturns DataFrame Shape: {returns_df.shape}")
    print(f"Date Range: {returns_df.index.min()} to {returns_df.index.max()}")


In [131]:
def hrp_portfolio(returns_df):
    """
    Compute HRP portfolio allocations for a DataFrame of asset returns.

    Args:
        returns_df (pd.DataFrame): DataFrame with dates as rows, assets as columns,
                                   and daily returns as values (e.g., shape: (n_days, n_assets)).

    Returns:
        pd.DataFrame: Portfolio weights for each asset (index: asset names, column: 'weights').
    """
    # Step 1: Compute the covariance matrix and convert to NumPy array
    cov_matrix = returns_df.cov().values  # Convert to NumPy array
    
    # Ensure positive definiteness
    min_eig = np.min(np.real(np.linalg.eigvals(cov_matrix)))
    if min_eig <= 0:
        cov_matrix += (abs(min_eig) + 1e-5) * np.eye(cov_matrix.shape[0])

    # Step 2: Compute the correlation-based distance matrix
    std = np.sqrt(np.diag(cov_matrix))
    std[std == 0] = 1e-10  # Prevent division by zero
    corr = cov_matrix / np.outer(std, std)
    corr = np.clip(corr, -1, 1)  # Clip to valid range
    np.fill_diagonal(corr, 1)    # Ensure diagonal is 1 (now works since corr is a NumPy array)
    corr = (corr + corr.T) / 2   # Force symmetry

    distance = np.sqrt(np.maximum(0.5 * (1 - corr), 0))
    np.fill_diagonal(distance, 0)  # Ensure diagonal is 0
    distance = (distance + distance.T) / 2  # Force symmetry

    # Step 3: Perform hierarchical clustering
    dist_condensed = squareform(distance)
    clusters = linkage(dist_condensed, method='single')

    # Step 4: Allocate weights using HRP
    assets = returns_df.columns
    n = len(assets)
    weights = pd.Series(1.0, index=assets)
    clusters_dict = {i: [i] for i in range(n)}

    for i in range(n - 1):
        cluster1 = clusters_dict[int(clusters[i, 0])]
        cluster2 = clusters_dict[int(clusters[i, 1])]
        
        # Compute cluster variances (convert to NumPy array for slicing)
        cov_matrix_df = returns_df.cov()  # Keep as DataFrame for iloc indexing
        cluster_cov1 = cov_matrix_df.iloc[cluster1, cluster1].values
        cluster_cov2 = cov_matrix_df.iloc[cluster2, cluster2].values
        w1 = np.ones(len(cluster1)) / len(cluster1)
        w2 = np.ones(len(cluster2)) / len(cluster2)
        var1 = max(w1.T @ cluster_cov1 @ w1, 1e-10)
        var2 = max(w2.T @ cluster_cov2 @ w2, 1e-10)
        
        # Allocate weights inversely proportional to variance
        weight1 = 1 / var1
        weight2 = 1 / var2
        alpha = weight1 / (weight1 + weight2)
        
        weights.iloc[cluster1] *= alpha
        weights.iloc[cluster2] *= (1 - alpha)
        
        # Update clusters
        new_cluster_id = n + i
        clusters_dict[new_cluster_id] = cluster1 + cluster2

    # Normalize weights
    weights = weights / weights.sum()
    return pd.DataFrame(weights, columns=['weights'])

In [ ]:
hrp_portfolio(returns_df).sort_values(by='weights', ascending=False).head(10)